# Verified GitHub → Hugging Face → Colab training path

This notebook proves the complete engineering chain with the first-place solution's
Ettin-400M component:

1. clone public code from GitHub;
2. install pinned dependencies without replacing Colab's CUDA Torch;
3. authenticate with Colab Secrets;
4. download Kaggle data;
5. download a pinned Hugging Face model revision;
6. run a real 32-row smoke training;
7. run full training and inference;
8. persist the model, logs, manifest, and submission to Google Drive.

Required Secret: `KAGGLE_API_TOKEN`. Recommended Secret: read-only `HF_TOKEN`.

In [1]:
REPOSITORY = "mingzhuoFUN/jigsaw-agile-community-rules"
BRANCH = "main"
WORKDIR = "/content/jigsaw-verified"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/jigsaw-verified-ettin"
RUN_FULL_TRAINING = True

In [2]:
import os, shutil, subprocess

clone_url = f"https://github.com/{REPOSITORY}.git"
if os.path.isdir(f"{WORKDIR}/.git"):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=True)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, WORKDIR], check=True)
os.chdir(WORKDIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

Commit: 04b824430a0ab509a1be28f9c2a9c309114796a6


In [3]:
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if not torch.cuda.is_available() or "+cpu" in torch.__version__:
    raise RuntimeError("Select a GPU runtime, delete the current runtime, and start again.")
print("GPU:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128 cuda: True
GPU: Tesla T4


In [4]:
%pip install -q -r requirements-verified-colab.txt
%pip install -q -e . --no-deps

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 8.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jigsaw-first-place-reproduction (pyproject.toml) ... done


In [5]:
import os, subprocess, sys
from pathlib import Path

src_path = str(Path(WORKDIR) / "src")
os.environ["PYTHONPATH"] = os.pathsep.join(
    [src_path, os.environ.get("PYTHONPATH", "")]
).rstrip(os.pathsep)
if src_path not in sys.path:
    sys.path.insert(0, src_path)
subprocess.run([sys.executable, "-m", "compileall", "-q", "src", "scripts"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
print("Repository import and tests passed.")

Repository import and tests passed.


In [6]:
import os
from google.colab import userdata

kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise ValueError("Add KAGGLE_API_TOKEN to Colab Secrets.")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face authenticated.")
else:
    print("HF_TOKEN is absent; public anonymous download will be used.")

Hugging Face authenticated.


In [7]:
from pathlib import Path
import subprocess, zipfile

data_dir = Path(WORKDIR) / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)
archive = data_dir / "jigsaw-agile-community-rules.zip"
if not (data_dir / "train.csv").exists():
    subprocess.run([
        "kaggle", "competitions", "download",
        "-c", "jigsaw-agile-community-rules", "-p", str(data_dir)
    ], check=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(data_dir)
required = {"train.csv", "test.csv", "sample_submission.csv"}
assert required <= {path.name for path in data_dir.glob("*.csv")}
print("Competition data ready:", data_dir)

Competition data ready: /content/jigsaw-verified/data/raw


In [8]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
output_dir = Path(DRIVE_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print("Persistent output:", output_dir)

Mounted at /content/drive
Persistent output: /content/drive/MyDrive/jigsaw-verified-ettin


In [9]:
# Real low-cost training gate; downloads the same pinned HF model used below.
smoke_dir = Path("/content/jigsaw-ettin-smoke")
subprocess.run([
    sys.executable, "scripts/run_verified_ettin.py",
    "--data-dir", str(data_dir),
    "--output-dir", str(smoke_dir),
    "--smoke-rows", "32",
], check=True)
assert (smoke_dir / "submission7.csv").exists()
print("Smoke training passed.")

Smoke training passed.


In [10]:
if RUN_FULL_TRAINING:
    subprocess.run([
        sys.executable, "scripts/run_verified_ettin.py",
        "--data-dir", str(data_dir),
        "--output-dir", str(output_dir),
    ], check=True)
    required = [
        output_dir / "submission7.csv",
        output_dir / "training.log",
        output_dir / "run_manifest.json",
        output_dir / "model" / "config.json",
    ]
    for path in required:
        assert path.exists() and path.stat().st_size > 0, path
    print("Full remote training complete.")
    for path in required:
        print(path, path.stat().st_size)
else:
    print("Smoke passed. Set RUN_FULL_TRAINING=True to execute full training.")

Full remote training complete.
/content/drive/MyDrive/jigsaw-verified-ettin/submission7.csv 122
/content/drive/MyDrive/jigsaw-verified-ettin/training.log 3718
/content/drive/MyDrive/jigsaw-verified-ettin/run_manifest.json 333
/content/drive/MyDrive/jigsaw-verified-ettin/model/config.json 1354


In [11]:
from pathlib import Path

model_dir = Path("/content/drive/MyDrive/jigsaw-verified-ettin/model")

for path in sorted(model_dir.rglob("*")):
    if path.is_file():
        print(f"{path.name:40s} {path.stat().st_size / 1024**2:.2f} MB")

config.json                              0.00 MB
model.safetensors                        1510.00 MB
special_tokens_map.json                  0.00 MB
tokenizer.json                           3.42 MB
tokenizer_config.json                    0.02 MB
